In [ ]:
import sqlite3

connection = sqlite3.connect("catalog.db")
cursor = connection.cursor()

In [ ]:
cursor.execute("""
    CREATE TABLE decor_items (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,

    -- core identity
    name            TEXT NOT NULL,
    category        TEXT NOT NULL,        -- Eg:'rug', 'lamp', 'shelf', 'wall_art'...
    retailer        TEXT NOT NULL,
    source_url      TEXT NOT NULL,
    image_url       TEXT,

    -- structured filter fields (hard constraints)
    price           REAL,
    currency        TEXT DEFAULT 'USD',
    width_cm        REAL,
    height_cm       REAL,
    depth_cm        REAL,
    color           TEXT,                  -- primary color, simplified

    -- semantic fields (what gets embedded)
    description     TEXT,                  -- raw product description
    style_tags      TEXT,                  -- JSON array as text: '["minimalist","mid-century"]'

    -- embedding cache (computed once at index time)
    embedding       BLOB,                  -- serialized vector, e.g. float32 numpy array

    -- freshness / lifecycle
    in_stock        INTEGER DEFAULT 1,     -- boolean as 0/1
    last_seen_at    TEXT NOT NULL,         -- ISO timestamp, updated every refresh run
    first_seen_at   TEXT NOT NULL,
    is_active       INTEGER DEFAULT 1      -- soft-delete flag, see refresh logic below
    );
""")

In [ ]:
cursor.execute("""
    CREATE INDEX idx_category ON decor_items(category);
    """)
cursor.execute("""
    CREATE INDEX idx_price ON decor_items(price);
    """)
cursor.execute("""
    CREATE INDEX idx_active ON decor_items(is_active);
    """)
cursor.execute("""
    CREATE UNIQUE INDEX idx_source_url ON decor_items(source_url);
    """)


In [27]:
from datetime import datetime, timezone
now = datetime.now(timezone.utc).isoformat()

In [35]:
def upsert_items(cursor, item):
    cursor.execute("""
        INSERT INTO decor_items (name, category, retailer, source_url, image_url, price, currency, width_cm, height_cm, depth_cm, color, description, style_tags, embedding, in_stock, last_seen_at, first_seen_at, is_active)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1)
        ON CONFLICT(source_url) DO UPDATE SET
            price = excluded.price,
            in_stock = excluded.in_stock,
            last_seen_at = excluded.last_seen_at,
            is_active = 1""",
            (item["name"], item["category"], item["retailer"], item["source_url"],
            item.get("image_url"), item.get("price"), item.get("currency", "USD"),
            item.get("width_cm"), item.get("height_cm"), item.get("depth_cm"),
            item.get("color"), item.get("description"), item.get("style_tags"),
            item.get("embedding"), item.get("in_stock", 1), item.get("last_seen_at"), item.get("first_seen_at")
        ))
    cursor.execute(
        """
        UPDATE decor_items
        SET is_active = 0
        WHERE last_seen_at < ? AND is_active = 1;
        """,
        (item.get("last_seen_at"),)
    )

In [32]:
print(now)

2026-06-20T10:27:59.990855+00:00


In [36]:
test_items = [
    {
        "name": "Oakwood Floor Lamp",
        "category": "lamp",
        "retailer": "TestStore",
        "source_url": "https://example.com/oakwood-lamp",
        "price": 89.0,
        "width_cm": 30,
        "height_cm": 150,
        "depth_cm": 30,
        "color": "natural wood",
        "description": "Minimalist oak floor lamp with linen shade.",
        "style_tags": '["minimalist", "scandinavian"]',
        "last_seen_at": now,
        "first_seen_at": now,
    },
    {
        "name": "Terracotta Planter Set",
        "category": "planter",
        "retailer": "TestStore",
        "source_url": "https://example.com/terracotta-set",
        "price": 34.5,
        "width_cm": 18,
        "height_cm": 20,
        "depth_cm": 18,
        "color": "terracotta",
        "description": "Set of 3 handmade terracotta planters.",
        "style_tags": '["rustic", "bohemian"]',
        "last_seen_at": now,
        "first_seen_at": now,
    },
]

for item in test_items:
    upsert_items(cursor, item)
connection.commit()